# Trip durations: from a static median to a model

The ride app quotes every trip at **12.25 minutes**, the median of Day 1's trips. Product wants a real prediction before the rider books, plus an assistant that can explain it.

This notebook is where you look at the data and try models. It is **not** where models are made: the pipeline (`pipelines/src/labpipes/defs/training.py`) trains, evaluates and registers them, and only it can set the `challenger` alias. Deciding what stays here and what moves into the pipeline is decision record 007.

Runs on the lab's SageMaker notebook (`conda_python3`). Settings come from `~/SageMaker/.lab.env`, written when the notebook starts.

In [ ]:
import os, sys, time
from pathlib import Path

env = Path.home() / "SageMaker" / ".lab.env"
for line in env.read_text().splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ.setdefault(k, v)

# The repo the notebook cloned (written by the start-up script), else the folder above this notebook.
REPO = Path(os.environ.get("LAB_REPO_DIR") or Path.cwd().parent)
sys.path.insert(0, str(REPO / "pipelines" / "src"))  # tripmodel: the same models the pipeline trains
{k: os.environ[k] for k in ("MLFLOW_TRACKING_URI", "REDSHIFT_CLUSTER_ID", "REDSHIFT_DATABASE", "REDSHIFT_DB_USER")}

## Querying Redshift

Through the Data API as `labadmin`, with the notebook's LabRole: no password, no driver. Results come back as rows; keep samples small (the API returns at most 100 MB).

In [ ]:
import boto3
import pandas as pd

rsd = boto3.client("redshift-data", region_name="us-east-1")

def sql(query: str) -> pd.DataFrame:
    sid = rsd.execute_statement(Sql=query, ClusterIdentifier=os.environ["REDSHIFT_CLUSTER_ID"],
                                Database=os.environ["REDSHIFT_DATABASE"], DbUser=os.environ["REDSHIFT_DB_USER"])["Id"]
    while (d := rsd.describe_statement(Id=sid))["Status"] not in ("FINISHED", "FAILED", "ABORTED"):
        time.sleep(1)
    if d["Status"] != "FINISHED":
        raise RuntimeError(d.get("Error"))
    if not d.get("HasResultSet"):
        return pd.DataFrame()
    rows, cols, token = [], None, None
    while True:
        page = rsd.get_statement_result(Id=sid, **({"NextToken": token} if token else {}))
        cols = [c["name"] for c in page["ColumnMetadata"]]
        rows += [[None if f.get("isNull") else next(iter(f.values())) for f in r] for r in page["Records"]]
        if not (token := page.get("NextToken")):
            return pd.DataFrame(rows, columns=cols)

sql("select trip_month, count(*) as trips from staging.trip_features group by 1 order by 1")

## What durations look like

A 1% sample of one month is plenty to see the shape. Which features does the contract allow (`contracts/features.yaml`), and why isn't the trip's own distance one of them?

In [ ]:
sample = sql("""
    select pickup_zone_id, dropoff_zone_id, pickup_hour, pickup_dow, duration_min
    from staging.trip_features where trip_month = '2024-01' and random() < 0.01""")
sample[["duration_min"]].describe(percentiles=[.1, .5, .9, .99]).T

In [ ]:
by_hour = sample.groupby(["pickup_dow", "pickup_hour"])["duration_min"].median().unstack()
by_hour.index = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
by_hour.round(0)  # median minutes by day and pickup hour

In [ ]:
zones = sql("select location_id, borough, zone from ingest.taxi_zones")
top = sql("""select zone, pickup_hour, trip_count, median_duration_min, p90_duration_min
             from marts.zone_hour_stats where zone in ('JFK Airport', 'Times Sq/Theatre District')
             and pickup_hour in (8, 17) order by 1, 2""")
top

## Three models on the same holdout

The static median, v1 (median per zone pair and hour) and v2 (LightGBM), all from `tripmodel`, the package the pipeline uses. Train on three months, test on the month after. A sample keeps this to a minute or two on an `ml.t3.medium`; the pipeline trains on more.

In [ ]:
from tripmodel import BaselineModel, LightGBMModel, evaluate

def month(m, frac=0.05):
    return sql(f"""select pickup_zone_id, dropoff_zone_id, pickup_hour, pickup_dow, duration_min
                    from staging.trip_features where trip_month = '{m}' and random() < {frac}""")

train = pd.concat([month(m) for m in ("2024-01", "2024-02", "2024-03")], ignore_index=True)
holdout = month("2024-04", 0.02)
distances = sql("""select pickup_zone_id, dropoff_zone_id, median(trip_distance) as est_distance_miles
                    from staging.stg_trips where trip_month in ('2024-01', '2024-02', '2024-03')
                    and duration_min > 0 and duration_min <= 180 and trip_distance > 0 group by 1, 2""")
for df in (train, holdout):
    df[["pickup_zone_id", "dropoff_zone_id", "pickup_hour", "pickup_dow"]] = df[["pickup_zone_id", "dropoff_zone_id", "pickup_hour", "pickup_dow"]].astype("int64")
len(train), len(holdout), len(distances)

In [ ]:
boroughs = dict(zip(zones.location_id.astype(int), zones.borough))
static = float((holdout.duration_min - 12.25).abs().mean())
v1 = BaselineModel().fit(train, distances)
v2 = LightGBMModel().fit(train, distances, num_rounds=200)
results = pd.DataFrame({
    "v1 baseline": evaluate(lambda df: v1.predict(None, df), holdout, boroughs, latency_samples=50),
    "v2 lightgbm": evaluate(lambda df: v2.predict(None, df), holdout, boroughs, latency_samples=50),
}).T
results.insert(0, "static_mae_min", static)
results.round(2)

## Tracking an experiment

Notebook runs go to their own experiment so they never mix with the pipeline's. Nothing here registers a model: registration, the gate and the `challenger` alias belong to the pipeline.

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("trip-duration-notebook")
with mlflow.start_run(run_name="v1 vs v2, 5% sample"):
    mlflow.log_params({"train_rows": len(train), "holdout_rows": len(holdout), "num_rounds": 200})
    for name, row in results.iterrows():
        mlflow.log_metrics({f"{name.split()[0]}.{k}": float(v) for k, v in row.items()})
print("open MLflow through SSM port forwarding: deploy/compose.platform.yaml")

## Before you leave the notebook (decision record 007)

- Which of these steps belong in the pipeline, run on every new month without you, and which are exploration only?
- What does the v1 baseline tell you about how much the model must beat it by (record 008)?
- Three months to train, the next one to test: is that the right window (record 009)? What would a longer one cost and buy?